# Day 1: Representation as Measurement

The goal today is to see how text becomes data. The same corpus can support different conclusions depending on the unit of analysis, preprocessing, feature weighting, and similarity measure.

By the end you should be able to:

1. Explain what spaCy contributes to an NLP workflow.
2. Build a document-term matrix.
3. Compare raw counts, binary counts, normalized counts, and TF-IDF.
4. Explain what common preprocessing choices remove or preserve.
5. Compute document similarity.
6. Compare groups using distinctive words.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', 120)

In [ ]:
from pathlib import Path


def find_data_dir():
    candidates = [Path('../data'), Path('materials/data'), Path('data')]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not find the course data directory.')


DATA_DIR = find_data_dir()
DATA_DIR

## spaCy setup

The notebooks use spaCy for tokenization and preprocessing. If `en_core_web_sm` is installed, the same setup also shows POS tags, lemmas, and dependency parses. If not, the notebooks still run with a blank English tokenizer.

In [ ]:
import spacy

try:
    nlp = spacy.load('en_core_web_sm')
    print('Loaded spaCy model: en_core_web_sm')
except OSError:
    nlp = spacy.blank('en')
    nlp.add_pipe('sentencizer')
    print('Using spacy.blank("en"). Install en_core_web_sm for POS tags, lemmas, dependency parses, and named entities.')


def spacy_preprocess(text, remove_stop=True, keep_alpha=True, min_len=2):
    """Tokenize and lightly preprocess text with spaCy."""
    doc = nlp.make_doc(str(text))
    tokens = []
    for token in doc:
        if keep_alpha and not token.is_alpha:
            continue
        if remove_stop and token.is_stop:
            continue
        term = token.text.lower()
        if len(term) >= min_len:
            tokens.append(term)
    return tokens


def spacy_analyzer(text):
    return spacy_preprocess(text)


def spacy_analyzer_with_bigrams(text):
    tokens = spacy_preprocess(text)
    bigrams = [tokens[i] + '_' + tokens[i + 1] for i in range(len(tokens) - 1)]
    return tokens + bigrams


def spacy_token_table(text):
    """Show tokenization, preprocessing flags, and parses when a full model is available."""
    doc = nlp(str(text))
    rows = []
    for token in doc:
        rows.append({
            'text': token.text,
            'lemma': token.lemma_ or '(model needed)',
            'pos': token.pos_ or '(model needed)',
            'dep': token.dep_ or '(model needed)',
            'head': token.head.text if token.dep_ else '(model needed)',
            'ent_type': token.ent_type_ or '',
            'is_alpha': token.is_alpha,
            'is_stop': token.is_stop,
            'like_num': token.like_num,
            'kept_by_preprocess': token.text.lower() in spacy_preprocess(token.text, remove_stop=True)
        })
    return pd.DataFrame(rows)

## 1. What spaCy does

spaCy is a practical NLP library for turning raw text into linguistic objects. In this course we mainly use it for preprocessing before statistical modeling, but the same pipeline can also support sentence segmentation, part-of-speech tagging, lemmatization, dependency parsing, named-entity recognition, rule-based matching, and feature extraction.

The important distinction is that spaCy gives us linguistic annotations; it does not, by itself, decide which representation is valid for a research question. We still choose what to count, remove, preserve, and validate.

In [ ]:
component_descriptions = {
    'tok2vec': 'contextual token features used by later components',
    'tagger': 'part-of-speech tags such as NOUN, VERB, ADJ',
    'parser': 'dependency parse and sentence boundaries',
    'attribute_ruler': 'rule-based token attributes and exceptions',
    'lemmatizer': 'dictionary/base forms such as studies -> study',
    'ner': 'named entities such as people, places, organizations',
    'sentencizer': 'rule-based sentence boundaries'
}

pipeline_table = pd.DataFrame({
    'component': ['tokenizer'] + list(nlp.pipe_names),
    'what_it_adds': ['splits text into tokens'] + [component_descriptions.get(name, 'pipeline component') for name in nlp.pipe_names]
})

capabilities = pd.DataFrame([
    {'capability': 'tokenization', 'available': True, 'typical_use': 'define words, punctuation, URLs, numbers'},
    {'capability': 'sentence segmentation', 'available': any(name in nlp.pipe_names for name in ['parser', 'senter', 'sentencizer']), 'typical_use': 'split speeches/articles into sentences'},
    {'capability': 'lemmatization', 'available': 'lemmatizer' in nlp.pipe_names, 'typical_use': 'collapse inflected forms'},
    {'capability': 'part-of-speech tags', 'available': 'tagger' in nlp.pipe_names, 'typical_use': 'keep nouns/verbs/adjectives or study style'},
    {'capability': 'dependency parsing', 'available': 'parser' in nlp.pipe_names, 'typical_use': 'inspect grammatical relations'},
    {'capability': 'named entities', 'available': 'ner' in nlp.pipe_names, 'typical_use': 'extract people, organizations, places, dates'}
])

display(pipeline_table)
display(capabilities)

### A sentence as a spaCy document

A spaCy `Doc` contains tokens plus optional annotations. The table below shows how one sentence is represented. If the full English model is installed, you will see lemmas, POS tags, dependency labels, syntactic heads, and named-entity types. With the blank fallback model, only tokenizer-level attributes are available.

In [ ]:
spacy_example = 'The European Commission proposed new regulations on bananas and chocolate!'
example_doc = nlp(spacy_example)

spacy_token_table(spacy_example)

In [ ]:
entity_rows = [
    {
        'entity': ent.text,
        'label': ent.label_,
        'start_token': ent.start,
        'end_token': ent.end
    }
    for ent in example_doc.ents
]

if entity_rows:
    display(pd.DataFrame(entity_rows))
else:
    print('No named entities available. Install en_core_web_sm to run the NER component.')

### Visualizing parses and entities

For teaching and error analysis, visual inspection is useful. spaCy's `displaCy` can render dependency parses and named entities directly inside a notebook. These views are diagnostics: they help us decide whether the preprocessing pipeline is appropriate for the texts we are studying.

In [ ]:
from IPython.display import HTML
from spacy import displacy

if 'parser' in nlp.pipe_names:
    dep_html = displacy.render(example_doc, style='dep', page=False, jupyter=False, options={'compact': True, 'distance': 90})
    display(HTML(dep_html))
else:
    print('Dependency visualization skipped because the parser is not available.')

if 'ner' in nlp.pipe_names and list(example_doc.ents):
    ent_html = displacy.render(example_doc, style='ent', page=False, jupyter=False)
    display(HTML(ent_html))
else:
    print('Entity visualization skipped because NER is not available or no entities were found.')

### Batch processing with `nlp.pipe`

For a few sentences, calling `nlp(text)` is fine. For hundreds or thousands of documents, `nlp.pipe()` is the standard spaCy pattern. It streams texts through the pipeline efficiently and returns one `Doc` per text.

In [ ]:
spacy_demo_texts = [
    'Courts blocked the policy after civil-rights groups filed a lawsuit.',
    'The minister announced a new climate package in Vienna.',
    'Researchers used transformer models to classify parliamentary speeches.',
    'Voters discussed inflation, housing, and migration during the campaign.'
]

demo_docs = list(nlp.pipe(spacy_demo_texts))
summary_rows = []
pos_rows = []
for doc_id, doc in enumerate(demo_docs, start=1):
    entities = ', '.join(f'{ent.text} ({ent.label_})' for ent in doc.ents) or '(none)'
    summary_rows.append({
        'doc_id': doc_id,
        'n_tokens': len(doc),
        'n_alpha_tokens': sum(token.is_alpha for token in doc),
        'n_sentences': sum(1 for _ in doc.sents),
        'entities': entities
    })
    for token in doc:
        if token.is_alpha:
            pos_rows.append({'doc_id': f'doc_{doc_id}', 'pos': token.pos_ or 'unknown'})

batch_summary = pd.DataFrame(summary_rows)
pos_counts = pd.DataFrame(pos_rows).value_counts(['doc_id', 'pos']).reset_index(name='tokens')

display(batch_summary)

plt.figure(figsize=(8, 4))
sns.barplot(data=pos_counts, x='tokens', y='pos', hue='doc_id')
plt.title('spaCy part-of-speech profile by example document')
plt.xlabel('Token count')
plt.ylabel('POS tag')
plt.tight_layout()

### Practical uses in text analysis projects

Common spaCy uses in applied text analysis:

1. Preprocessing: tokenize, lowercase, remove stop words, keep only alphabetic tokens, or lemmatize.
2. Feature engineering: count nouns, verbs, entities, sentence lengths, dependency labels, or custom patterns.
3. Quality control: inspect whether tokenization and sentence segmentation work for a corpus.
4. Information extraction: identify organizations, people, dates, locations, and domain-specific phrase patterns.
5. Scaling up: process many documents with `nlp.pipe()` before statistical modeling.

The rest of this notebook uses spaCy primarily as a transparent preprocessing layer before building document-term matrices.

## 2. A tiny corpus

Start with a toy example where every transformation is visible. This is not realistic, but it makes the representation choices inspectable.

In [ ]:
toy_texts = [
    'John loves ice cream and oranges.',
    'Mary loves chocolate ice cream.',
    'John dislikes chocolate but loves oranges.',
    'Mary hates oranges and loves chocolate.'
]

toy = pd.DataFrame({
    'doc_id': ['doc_1', 'doc_2', 'doc_3', 'doc_4'],
    'speaker': ['John', 'Mary', 'John', 'Mary'],
    'text': toy_texts
})
toy

## 3. From spaCy tokens to features

This table shows what spaCy sees in one toy sentence. With `en_core_web_sm`, the lemma, POS, dependency, syntactic head, and entity columns become real annotations; with the fallback tokenizer, tokenization and lexical flags still work.

The key modeling choice comes next: once spaCy has produced tokens, we decide which token attributes become columns in a document-term matrix.

In [ ]:
spacy_token_table(toy_texts[0])

In [ ]:
vectorizer = CountVectorizer(analyzer=spacy_analyzer)
X = vectorizer.fit_transform(toy['text'])

dtm = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out(), index=toy['doc_id'])
dtm

## 4. Representation choices

Compare several common representations. Ask what each one makes easy to see and what each one hides.

### Methodology formulas: representation as measurement

Most of the notebook turns raw text into a document-feature matrix. If document $d$ contains vocabulary item $v$, the count representation is

$$X_{dv} = c(v, d).$$

TF-IDF keeps local frequency but downweights terms that appear in many documents:

$$\mathrm{tfidf}_{dv} = \mathrm{tf}_{dv} \times \log\left(\frac{N}{\mathrm{df}_v}\right),$$

where $N$ is the number of documents and $\mathrm{df}_v$ is the number of documents containing term $v$. Similarity between two documents is usually measured by cosine similarity:

$$\cos(x_i, x_j) = \frac{x_i^\top x_j}{\lVert x_i \rVert_2\lVert x_j \rVert_2}.$$

For distinctive terms by group, the notebook uses a smoothed log-odds ratio. If $c_{v,A}$ is the count of term $v$ in group $A$, $N_A$ is the total number of term tokens in group $A$, and $\alpha$ is a small smoothing value, then

$$\mathrm{odds}_{v,A}=\frac{c_{v,A}+\alpha}{N_A-c_{v,A}+\alpha}, \qquad \delta_v=\log(\mathrm{odds}_{v,A})-\log(\mathrm{odds}_{v,B}).$$

A rough standard error for the contrast is

$$\mathrm{se}(\delta_v) \approx \sqrt{\frac{1}{c_{v,A}+\alpha}+\frac{1}{c_{v,B}+\alpha}}, \qquad z_v=\frac{\delta_v}{\mathrm{se}(\delta_v)}.$$

This lets us distinguish large but noisy rare-word contrasts from frequent, stable contrasts. The two-dimensional document map later in the notebook uses a low-rank approximation of the feature matrix:

$$X \approx U_k \Sigma_k V_k^\top.$$

In [ ]:
representations = {
    'spaCy tokens, raw counts': CountVectorizer(
        analyzer=lambda text: spacy_preprocess(text, remove_stop=False, keep_alpha=True, min_len=1)
    ),
    'spaCy tokens, binary word presence': CountVectorizer(
        analyzer=lambda text: spacy_preprocess(text, remove_stop=False, keep_alpha=True, min_len=1),
        binary=True
    ),
    'spaCy preprocessing: no stop words': CountVectorizer(analyzer=spacy_analyzer),
    'spaCy preprocessing + TF-IDF': TfidfVectorizer(analyzer=spacy_analyzer)
}

for name, vec in representations.items():
    matrix = vec.fit_transform(toy['text'])
    table = pd.DataFrame(matrix.toarray(), columns=vec.get_feature_names_out(), index=toy['doc_id'])
    print()
    print(name.upper())
    display(table.round(3))

## 5. Move to real data

We will use State of the Union speeches. The file has one row per speech, with president, year, party, and text.

In [ ]:
sotu = pd.read_csv(DATA_DIR / 'sotu.csv')
sotu['year_numeric'] = pd.to_numeric(sotu['year'], errors='coerce')

sotu[['president', 'year', 'party', 'sotu_type', 'text']].head()

In [ ]:
modern = (
    sotu
    .dropna(subset=['year_numeric', 'party', 'text'])
    .query("party in ['Democratic', 'Republican'] and year_numeric >= 1950")
    .copy()
)

modern['doc_label'] = modern['president'] + ' (' + modern['year'].astype(str) + ')'
modern[['doc_label', 'party', 'sotu_type']].head(), modern.shape

## 6. Most frequent terms

Frequency is simple and useful, but frequent words are not always the most substantively interesting words.

In [ ]:
count_vec = CountVectorizer(analyzer=spacy_analyzer, min_df=2, max_features=2000)
counts = count_vec.fit_transform(modern['text'])
terms = count_vec.get_feature_names_out()

term_counts = pd.Series(np.asarray(counts.sum(axis=0)).ravel(), index=terms)
term_counts.sort_values(ascending=False).head(20).to_frame('count')

In [ ]:
term_counts.sort_values(ascending=False).head(20).sort_values().plot(kind='barh', figsize=(7, 5))
plt.title('Most frequent terms in modern SOTU speeches')
plt.xlabel('Count')
plt.tight_layout()

## 6a. Frequency structure and document length

Advanced text analysis often starts with diagnostics. Zipf-style frequency plots and document-length distributions reveal whether a corpus is dominated by a few common terms or by uneven document sizes.

In [ ]:
modern['n_spacy_tokens'] = modern['text'].apply(
    lambda text: len(spacy_preprocess(text, remove_stop=False, keep_alpha=True, min_len=1))
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ranked_counts = term_counts.sort_values(ascending=False).reset_index(drop=True)
axes[0].loglog(np.arange(1, len(ranked_counts) + 1), ranked_counts.values)
axes[0].set_title('Zipf-style term frequency plot')
axes[0].set_xlabel('Term rank')
axes[0].set_ylabel('Frequency')

sns.histplot(data=modern, x='n_spacy_tokens', hue='party', bins=25, element='step', ax=axes[1])
axes[1].set_title('Document length by party')
axes[1].set_xlabel('Number of spaCy tokens')

plt.tight_layout()

### Additional demo: sparsity and vocabulary growth

Document-term matrices are mostly zeros. This demo makes that sparsity visible and shows how the observed vocabulary grows as we add speeches in chronological order.

In [ ]:
matrix_cells = counts.shape[0] * counts.shape[1]
sparsity = 1 - counts.nnz / matrix_cells

print(f'Document-term matrix shape: {counts.shape[0]} documents x {counts.shape[1]} terms')
print(f'Nonzero entries: {counts.nnz:,} of {matrix_cells:,}')
print(f'Sparsity: {sparsity:.1%}')

sample_rows = np.linspace(0, counts.shape[0] - 1, min(40, counts.shape[0]), dtype=int)
sample_cols = np.asarray(counts.sum(axis=0)).ravel().argsort()[::-1][:80]
sparsity_view = (counts[sample_rows][:, sample_cols] > 0).toarray()
visible_terms = terms[sample_cols]

seen_terms = set()
growth_rows = []
for order, (_, row) in enumerate(modern.sort_values('year_numeric').iterrows(), start=1):
    seen_terms.update(spacy_preprocess(row['text']))
    growth_rows.append({
        'speech_number': order,
        'year': row['year_numeric'],
        'vocabulary_size': len(seen_terms)
    })

vocab_growth = pd.DataFrame(growth_rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.heatmap(
    sparsity_view,
    cmap=['white', '#4C72B0'],
    cbar=False,
    xticklabels=visible_terms,
    yticklabels=False,
    ax=axes[0]
)
axes[0].set_title('Nonzero entries for top terms')
axes[0].set_xlabel('Terms ordered by corpus frequency')
axes[0].set_ylabel('Sampled documents')
axes[0].tick_params(axis='x', labelrotation=90, labelsize=7)

sns.lineplot(data=vocab_growth, x='year', y='vocabulary_size', marker='o', ax=axes[1])
axes[1].set_title('Vocabulary growth over SOTU speeches')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Cumulative unique preprocessed terms')

plt.tight_layout()

## 7. Distinctive terms by group

Here we compare Democratic and Republican speeches using a smoothed log-odds ratio. Positive values are more Democratic in this coding; negative values are more Republican.

The standard error and z-score are not a substitute for substantive validation, but they help avoid over-interpreting rare words that appear only a few times.

In [ ]:
count_table = pd.DataFrame(counts.toarray(), columns=terms)
count_table['_party_label'] = modern['party'].to_numpy()

group_counts = count_table.groupby('_party_label')[list(terms)].sum()
group_counts = group_counts.loc[['Democratic', 'Republican']]

dem_counts = group_counts.loc['Democratic']
rep_counts = group_counts.loc['Republican']
dem_total = dem_counts.sum()
rep_total = rep_counts.sum()

# Jeffreys-style smoothing keeps zero counts finite without overwhelming the data.
alpha = 0.5

dem_odds = (dem_counts + alpha) / (dem_total - dem_counts + alpha)
rep_odds = (rep_counts + alpha) / (rep_total - rep_counts + alpha)
log_odds = np.log(dem_odds) - np.log(rep_odds)
log_odds_se = np.sqrt(1 / (dem_counts + alpha) + 1 / (rep_counts + alpha))
log_odds_z = log_odds / log_odds_se

distinctive = pd.DataFrame({
    'term': terms,
    'dem_count': dem_counts.to_numpy(),
    'rep_count': rep_counts.to_numpy(),
    'total_count': (dem_counts + rep_counts).to_numpy(),
    'dem_vs_rep_log_odds': log_odds.to_numpy(),
    'log_odds_se': log_odds_se.to_numpy(),
    'z_score': log_odds_z.to_numpy()
})
# Backward-compatible alias for older teaching notes that used this column name.
distinctive['dem_vs_rep_log_ratio'] = distinctive['dem_vs_rep_log_odds']

pd.concat([
    distinctive.sort_values('dem_vs_rep_log_odds', ascending=False).head(12),
    distinctive.sort_values('dem_vs_rep_log_odds', ascending=True).head(12)
])[['term', 'dem_count', 'rep_count', 'total_count', 'dem_vs_rep_log_odds', 'z_score']]

## 7a. Visualizing distinctive terms

The ranked plot below makes the group contrast easier to audit than a table. Large positive values are more Democratic; large negative values are more Republican.

In [ ]:
top_dem = distinctive.sort_values('dem_vs_rep_log_odds', ascending=False).head(12)
top_rep = distinctive.sort_values('dem_vs_rep_log_odds', ascending=True).head(12)
plot_terms = pd.concat([top_rep, top_dem]).sort_values('dem_vs_rep_log_odds')

plt.figure(figsize=(8, 7))
colors = np.where(plot_terms['dem_vs_rep_log_odds'] > 0, '#4C72B0', '#C44E52')
plt.barh(plot_terms['term'], plot_terms['dem_vs_rep_log_odds'], color=colors)
plt.axvline(0, color='black', linewidth=1)
plt.title('Distinctive terms: Democratic vs. Republican SOTU speeches')
plt.xlabel('Smoothed log-odds ratio')
plt.tight_layout()

## 7b. Log-odds funnel plot

A funnel plot shows the log-odds contrast against term frequency. Rare terms can have extreme log-odds simply because they appear only a few times. Frequent terms need a stronger and more consistent group imbalance to move far away from zero.

The dashed curves are a rough 95% reference band under an equal-split approximation. Terms outside the band are not automatically substantively meaningful, but they are better candidates for interpretation than isolated rare words.

In [ ]:
funnel = distinctive.copy()
funnel['direction'] = 'inside rough 95% band'
funnel.loc[(funnel['z_score'] >= 1.96), 'direction'] = 'more Democratic'
funnel.loc[(funnel['z_score'] <= -1.96), 'direction'] = 'more Republican'

min_count = max(1, int(funnel['total_count'].min()))
max_count = max(10, int(funnel['total_count'].max()))
funnel_counts = np.geomspace(min_count, max_count, 250)
# Reference band if a term's counts were split about evenly across the two groups.
funnel_band = 1.96 * np.sqrt(4 / (funnel_counts + 2 * alpha))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

palette = {
    'more Democratic': '#4C72B0',
    'more Republican': '#C44E52',
    'inside rough 95% band': '#9A9A9A'
}

# Keep the point size visually useful, but avoid a separate size legend.
point_sizes = np.interp(
    np.log1p(funnel['total_count']),
    (np.log1p(funnel['total_count']).min(), np.log1p(funnel['total_count']).max()),
    (18, 130)
)

for direction in ['inside rough 95% band', 'more Democratic', 'more Republican']:
    subset = funnel[funnel['direction'].eq(direction)]
    axes[0].scatter(
        subset['total_count'],
        subset['dem_vs_rep_log_odds'],
        s=point_sizes[subset.index],
        c=palette[direction],
        alpha=0.55 if direction == 'inside rough 95% band' else 0.7,
        linewidths=0,
        label=direction
    )

axes[0].plot(funnel_counts, funnel_band, color='black', linestyle='--', linewidth=1)
axes[0].plot(funnel_counts, -funnel_band, color='black', linestyle='--', linewidth=1)
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_xscale('log')
axes[0].set_xlim(min_count * 0.90, max_count * 1.25)
axes[0].set_title('Log-odds funnel plot')
axes[0].set_xlabel('Total term count across both parties, log scale')
axes[0].set_ylabel('Democratic vs. Republican log-odds')
axes[0].legend(title='Direction', bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)

# Label a small set of interpretable terms: outside the band and not extremely rare.
label_candidates = funnel[
    funnel['direction'].ne('inside rough 95% band')
    & funnel['total_count'].ge(funnel['total_count'].quantile(0.55))
].copy()
label_terms = label_candidates.reindex(
    label_candidates['z_score'].abs().sort_values(ascending=False).head(8).index
)

for label_i, (_, row) in enumerate(label_terms.iterrows()):
    y_offset = 0.18 if label_i % 2 == 0 else -0.18
    axes[0].annotate(
        row['term'],
        xy=(row['total_count'], row['dem_vs_rep_log_odds']),
        xytext=(8, 10 if y_offset > 0 else -12),
        textcoords='offset points',
        fontsize=8,
        alpha=0.9,
        arrowprops={'arrowstyle': '-', 'color': '0.45', 'lw': 0.7}
    )

top_z = funnel.reindex(funnel['z_score'].abs().sort_values(ascending=False).head(20).index).sort_values('z_score')
bar_colors = np.where(top_z['z_score'] > 0, '#4C72B0', '#C44E52')
axes[1].barh(top_z['term'], top_z['z_score'], color=bar_colors)
axes[1].axvline(0, color='black', linewidth=1)
axes[1].axvline(1.96, color='gray', linestyle='--', linewidth=1)
axes[1].axvline(-1.96, color='gray', linestyle='--', linewidth=1)
axes[1].set_title('Most distinctive terms by standardized log-odds')
axes[1].set_xlabel('z-score')
axes[1].set_ylabel('')

plt.tight_layout()

display(
    top_z[['term', 'dem_count', 'rep_count', 'total_count', 'dem_vs_rep_log_odds', 'z_score']]
    .sort_values('z_score', ascending=False)
    .round(3)
)

## 8. Document similarity

A vector space lets us ask which speeches are similar. Similarity is a property of the representation, not just the original texts.

In [ ]:
tfidf_vec = TfidfVectorizer(analyzer=spacy_analyzer, min_df=2, max_features=3000)
tfidf = tfidf_vec.fit_transform(modern['text'])
similarity = cosine_similarity(tfidf)

pairs = []
labels = modern['doc_label'].to_list()
parties = modern['party'].to_list()

for i in range(len(labels)):
    for j in range(i + 1, len(labels)):
        pairs.append({
            'doc_1': labels[i],
            'party_1': parties[i],
            'doc_2': labels[j],
            'party_2': parties[j],
            'cosine_similarity': similarity[i, j]
        })

pd.DataFrame(pairs).sort_values('cosine_similarity', ascending=False).head(10)

## 8a. Similarity heatmap

Similarity matrices are useful for checking whether documents cluster by time, speaker, party, genre, or some artifact of preprocessing.

In [ ]:
recent = modern.sort_values('year_numeric').tail(16).copy()
recent_tfidf = tfidf_vec.transform(recent['text'])
recent_similarity = cosine_similarity(recent_tfidf)

plt.figure(figsize=(10, 8))
sns.heatmap(
    recent_similarity,
    xticklabels=recent['doc_label'],
    yticklabels=recent['doc_label'],
    cmap='viridis',
    vmin=0,
    vmax=1
)
plt.title('Cosine similarity among recent SOTU speeches')
plt.xticks(rotation=80, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

## 8b. Representation geometry

A two-dimensional projection is not a model result by itself. It is a diagnostic view of how the chosen representation arranges documents.

In [ ]:
svd2 = TruncatedSVD(n_components=2, random_state=42)
coords = svd2.fit_transform(tfidf)

plot_df = modern[['president', 'year_numeric', 'party', 'sotu_type']].copy()
plot_df['x'] = coords[:, 0]
plot_df['y'] = coords[:, 1]

plt.figure(figsize=(8, 6))
sns.scatterplot(data=plot_df, x='x', y='y', hue='party', style='sotu_type', alpha=0.85)
plt.title('SVD projection of TF-IDF document vectors')
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.tight_layout()

## 9. Student task

Pick one change below and rerun the analysis:

- Change `min_df` from 2 to 5.
- Use binary word presence instead of counts.
- Include bigrams by using `spacy_analyzer_with_bigrams` as the vectorizer analyzer.
- Compare speeches before and after 1980.

Write three sentences: what changed, why did it change, and which representation best fits the research question?

In [ ]:
# Your turn: modify the vectorizer or the grouping variable here.
student_vec = CountVectorizer(analyzer=spacy_analyzer, min_df=2)
# For bigrams, try:
# student_vec = CountVectorizer(analyzer=spacy_analyzer_with_bigrams, min_df=2)

student_X = student_vec.fit_transform(modern['text'])

print(student_X.shape)
pd.Series(
    np.asarray(student_X.sum(axis=0)).ravel(),
    index=student_vec.get_feature_names_out()
).sort_values(ascending=False).head(20)